In [1]:
import torch
from torch.utils.data import Dataset, DataLoader

In [2]:
with open("data/tinyshakespeare/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

split = int(0.9 * len(text))

train_text = text[:split]
val_text = text[split:]

print(len(train_text), len(val_text))

1003854 111540


In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

def decode(tokens):
    return ''.join(itos[i] for i in tokens)
print(vocab_size)


65


In [4]:
train_data = torch.tensor([stoi[c] for c in train_text], dtype=torch.long)
val_data   = torch.tensor([stoi[c] for c in val_text], dtype=torch.long)

class ShakespeareDataset(Dataset):
    def __init__(self, data, seq_len) -> None:
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, index):
        x = self.data[index:index + self.seq_len] # the current letter
        y = self.data[index+ 1: index + self.seq_len + 1] # the next letter

        return x, y


seq_len = 100
batch_size = 64

train_dataset = ShakespeareDataset(train_data, seq_len)
val_dataset = ShakespeareDataset(val_data, seq_len)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [5]:
batch_size = 64
block_size = seq_len

n_embed = 128
n_head = 4
n_layer = 4

dropout = 0.1
learning_rate = 3e-4

In [6]:
from torch import nn
from torch.nn import functional as F

class AttentionHead(nn.Module):
    def __init__(self,head_size) -> None:
        super().__init__()
        self.key = nn.Linear(n_embed,head_size,bias=False)
        self.query = nn.Linear(n_embed,head_size,bias=False)
        self.value = nn.Linear(n_embed,head_size,bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))
        self.head_size = head_size

    def forward(self,x):
        B,T,C = x.shape
        K = self.key(x)
        Q = self.query(x)

        wei = Q @ K.transpose(-2,-1) * self.head_size ** -0.5
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei = F.softmax(wei,dim = -1)

        v = self.value(x)
        out = wei @ v

        return out

```
Head 1 ─┐
Head 2 ─┤
Head 3 ─┼→ concatenate → Wᴼ → output
Head 4 ─┘
```

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads,n_embd,head_size) -> None:
        super().__init__()
        self.heads = nn.ModuleList([
            AttentionHead(head_size)
            for _ in range(num_heads)
        ])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        out = torch.cat(
            [h(x) for h in self.heads ], dim = -1
        )
        out = self.proj(out)
        out = self.dropout(out)
        return out


In [8]:
class FeedForward(nn.Module):
    def __init__(self,n_embed) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed,n_embed * 4),
            nn.GELU(),
            nn.Linear(n_embed * 4,n_embed),
            nn.Dropout(dropout)
        )

    def forward(self,x):
        return self.net(x)

```
x
│
├── LayerNorm
│
├── Attention
│
└── + original x
       ↓
      x
│
├── LayerNorm
│
├── FFN
│
└── + original x
       ↓
     output
```

In [9]:
class Block(nn.Module):
    def __init__(self,n_embed,n_head) -> None:
        super().__init__()
        head_size = n_embed // n_head

        self.sa = MultiHeadAttention(n_head,n_embed,head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self,x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [10]:
class Transformer(nn.Module):
    def __init__(self,n_embed,n_head,n_layer) -> None:
        super().__init__()
        self.blocks = nn.Sequential(
            *[ Block(n_embed,n_head) for _ in range(n_layer)]
        )
        self.token_embedding = nn.Embedding(vocab_size,n_embed)
        self.positional_embedding = nn.Embedding(block_size,n_embed)
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed,vocab_size)

    def forward(self,x, targets=None):
        B,T = x.shape
        token_embed = self.token_embedding(x)

        positions = torch.arange(T, device=x.device)
        position_embed = self.positional_embedding(positions)

        x = token_embed + position_embed

        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B,T,C = logits.shape

            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B*T)

            loss = F.cross_entropy(logits_flat,targets_flat)
        return logits, loss

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Transformer(n_embed,n_head,n_layer).to(device)


In [12]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

In [13]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    total_train_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        logits, loss = model(xb, yb)

        optimizer.zero_grad(set_to_none=True)

        loss.backward()

        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = (
        total_train_loss / len(train_loader)
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"| Train Loss: {avg_train_loss:.4f}"
    )

Epoch 1/10 | Train Loss: 1.5024
Epoch 2/10 | Train Loss: 1.2720
Epoch 3/10 | Train Loss: 1.2171
Epoch 4/10 | Train Loss: 1.1824
Epoch 5/10 | Train Loss: 1.1565
Epoch 6/10 | Train Loss: 1.1355
Epoch 7/10 | Train Loss: 1.1177
Epoch 8/10 | Train Loss: 1.1024
Epoch 9/10 | Train Loss: 1.0893
Epoch 10/10 | Train Loss: 1.0784


In [14]:
model.eval()

total_loss = 0
total_batches = 0

with torch.no_grad():

    for xb, yb in val_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        logits, loss = model(xb, yb)

        total_loss += loss.item()
        total_batches += 1

val_loss = total_loss / total_batches
val_loss

1.6318257161930425

In [16]:
model.eval()

context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

with torch.no_grad():

    for _ in range(1000):

        # Keep context within block_size
        context_cond = context[:, -block_size:]

        logits, _ = model(context_cond)

        # Get logits for the final token
        logits = logits[:, -1, :]

        # Convert logits to probabilities
        probs = F.softmax(
            logits,
            dim=-1
        )

        # Sample next token
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        # Add new token
        context = torch.cat(
            (context, next_token),
            dim=1
        )

print(
    decode(
        context[0].tolist()
    )
)


MENENIUS:
Now be resolved to prick from me! This fair stand, much no world,
That Romeo, his prick son, wheready is call'd
Upon the mother, make doubtless.

MENENIUS:
Let's fear
Within all night virtues: be ridege: there is no day
When I was bowels, they jest--was deads.
You'ld by the duke wish me to-night?

KING RICHARD II:
I do this leave you to serve him to him in;
And what refrailed he were they, cheers, fear:
You will oath been to be on their power; woes are
Even too fond of a world: though they deposed,
It had raven the power of men was not so.

ROMEO:
O, they set down thing I both of the women's?
Ha! down this ancient, my lord, and my body
Cut the gwon of his white hath been been an as!
And thou wilt not pluck from whose trrick and bluster?

DUKE OF YORK:
Go, know you well.

HENRY BOLINGBROKE:
Faith, good my lord, and set that I smell mode:
You shall denied thee for your bed, blest:
Scratches of whose very gifts holy tainted lips
That knaves Edward oor and grievous Petruchio's h